#### **Problem Statement**

- Recent times have seen an increase in the use of large language models such as Gemini, ChatGPT, etc., for various applications.

- However, rapid deployment of these LLMs has raised concerns regarding their risks - including ethical issues, data biases, privacy and robustness. [1]

- AI Guardrails are safety mechanisms filtering the inputs or outputs of large language models (LLMs) confining the LLMs, *to provide a more secure range of function as well as compliance to ethical principles and mitigating hallucinations*. [1]

- Guardrails identify potential misuse in the query stage and try to prevent the LLM from providing a response which should not be given. [1]

- Examples of opensource guardrail systems are from GuardrailsAI, Llama Guard, NVIDIA's NeMO Guardrails.

- *Llama Guard developed by Meta, is based on the Llama2-7b architecture, focusing on human-AI conversation safety [1]* Llama guard classifies content in both LLM inputs (prompt classfication) and in LLM response (response classification). Generates text as part of the output, indicating whether a given prompt is safe as well potential category of safety violation. [3]

- *GuardrailsAI enables the user to add a structure, type and quality, defining the RAIL spec in XML or pydantic, defining "guard" and wrapping the LLM.*

- *NVIDIA NeMo Guardrail functions as an intermediary layer, enhancing the control and safety of LLM through the use of Colang.*

**References :**
1.  https://arxiv.org/pdf/2402.01822.pdf
2. https://attri.ai/blog
3. https://huggingface.co/meta-llama/LlamaGuard-7b
4. https://github.com/guardrails-ai/guardrails
5. https://www.guardrailsai.com/docs/guardrails_ai/getting_started


#### **Installing Dependencies**

In [ ]:
# Installing Dependencies : Openai, guardrails-ai, typing, pydantic
# Pinned versions (tested Oct 2026) so this notebook keeps working on Google Colab
!pip install -q openai==0.28.1 guardrails-ai==0.3.2 pydantic==2.4.2 tiktoken==0.5.2 rich
# !pip install openai==0.28 -q   # (replaced by the pinned install above)
# !pip install guardrails-ai -q   # (replaced by the pinned install above)
# !pip install rich -q   # (replaced by the pinned install above)
# !pip install typing -q   # (replaced by the pinned install above)
# !pip install tiktoken -q   # (replaced by the pinned install above)
# !pip install pydantic -q   # (replaced by the pinned install above)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.5/76.5 kB 1.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.5/176.5 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.1/113.1 kB 8.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.7/111.7 kB 7.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.8/250.8 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.7/105.7 kB 10.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.8/395.8 kB 15.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.3/110.3 kB 10.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 26.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 8.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━

#### **OPENAI API KEY**

In [ ]:
# Importing the required dependencies :
from google.colab import userdata # userdata from google.colab to store the OpenAI API KEY
from pydantic import BaseModel, Field # Importing BaseModel and Field from pydantic
from typing import List # Importing List from typing module
import openai # Importing openai
import json # library to work with json data

In [ ]:
# Storing the OpenAI API Key in google colab
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
openai.api_key = OPENAI_API_KEY

#### **Extracting information from text : Pydantic Model**

- Extracting information from the text : age, symptoms, gender, prescribed medication.

In [ ]:
from guardrails.validators import ValidRange, ValidChoices # Importing guardrails.validators : ValidRange and ValidChoices
import guardrails as gd  # Importing guardrails


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


In [ ]:
# Prompt for the model and defining the Pydantic classes :

prompt = """

Only with the following doctor's notes about a patient shown below in three backticks,
please extract a dictionary that contains the patient's information.

``` Patient is 35 years of age and is a Female, with mild symptoms consistent with allergic common cold.
Patient reports onset of symptoms three days ago, has an itchy nose, mild sneezing and occasional coughing.
No fever reported.
Prescribed over-the-counter cold remedy medication, showing slight improvement```

${gr.complete_json_suffix_v2}

"""

class Symptom(BaseModel):
  symptom: str = Field(
      description = "Symptoms that the patient is experiencing")
  affected_area: str = Field(
      description = "What part of the body is affected", validators=[ValidChoices(choices=['head', 'neck', 'nose', 'chest'], on_fail="reask")]
      )

class Medication(BaseModel):
  medication: str = Field(
      description = "Name of the medication prescribed to the patient")
  response: str = Field(
      description = "How is the patient responding to the prescribed medication"
      )

class PatientInfo(BaseModel):
  age : int = Field(
      description = "Patient's Age", validators=[ValidRange(min=0, max=100, on_fail="fix")]
      )
  gender: str = Field(
      description = "Patient's gender"
      )
  symptoms: List[Symptom] = Field(
      description = "Symptoms that the patient is currently experiencing. Each symptom should be classified into a separate item in the list."
      )
  current_meds: List[Medication] = Field(
      description = "Medications the patient is currently prescribed and their response."
      )


In [ ]:
# defining the guard object for validating and correcting the response from the large language model
guard = gd.Guard.from_pydantic(output_class = PatientInfo, prompt = prompt)

In [ ]:
#display the final prompt
print(guard.base_prompt)



Only with the following doctor's notes about a patient shown below in three backticks,
please extract a dictionary that contains the patient's information.

``` Patient is 35 years of age and is a Female, with mild symptoms consistent with allergic common cold.
Patient reports onset of symptoms three days ago, has an itchy nose, mild sneezing and occasional coughing.
No fever reported.
Prescribed over-the-counter cold remedy medication, showing slight improvement```


Given below is XML that describes the information to extract from this document and the tags to extract it into.

<output>
    <integer name="age" description="Patient's Age" format="valid-range: min=0 max=100"/>
    <string name="gender" description="Patient's gender"/>
    <list name="symptoms" description="Symptoms that the patient is currently experiencing. Each symptom should be classified into a separate item in the list.">
        <object>
            <string name="symptom" description="Symptoms that the patient 

In [ ]:
# Extracting response from the LLM :
response = guard(
    openai.ChatCompletion.create,
    model = "gpt-4o-mini",
    temperature = 0.3,
    max_tokens = 2048
)

In [ ]:
#display validated_output
print(json.dumps(response.validated_output, indent=2))

{
  "age": 35,
  "gender": "Female",
  "symptoms": [
    {
      "symptom": "itchy nose",
      "affected_area": "nose"
    },
    {
      "symptom": "mild sneezing",
      "affected_area": "head"
    },
    {
      "symptom": "occasional coughing",
      "affected_area": "chest"
    }
  ],
  "current_meds": [
    {
      "medication": "over-the-counter cold remedy",
      "response": "slight improvement"
    }
  ]
}


In [ ]:
# Display the output using guard.history
from rich import print
print(guard.history.last.iterations.first.rich_group)

╭──────────────────────────────────────────────────── Prompt ─────────────────────────────────────────────────────╮
│                                                                                                                 │
│                                                                                                                 │
│ Only with the following doctor's notes about a patient shown below in three backticks,                          │
│ please extract a dictionary that contains the patient's information.                                            │
│                                                                                                                 │
│ ``` Patient is 35 years of age and is a Female, with mild symptoms consistent with allergic common cold.        │
│ Patient reports onset of symptoms three days ago, has an itchy nose, mild sneezing and occasional coughing.     │
│ No fever reported.                                                                                              │
│ Prescribed over-the-counter cold remedy medication, showing slight improvement```                               │
│                                                                                                                 │
│                                                                                                                 │
│ Given below is XML that describes the information to extract from this document and the tags to extract it      │
│ into.                                                                                                           │
│                                                                                                                 │
│ <output>                                                                                                        │
│     <integer name="age" description="Patient's Age" format="valid-range: min=0 max=100"/>                       │
│     <string name="gender" description="Patient's gender"/>                                                      │
│     <list name="symptoms" description="Symptoms that the patient is currently experiencing. Each symptom should │
│ be classified into a separate item in the list.">                                                               │
│         <object>                                                                                                │
│             <string name="symptom" description="Symptoms that the patient is experiencing"/>                    │
│             <string name="affected_area" description="What part of the body is affected" format="valid-choices: │
│ choices=['head', 'neck', 'nose', 'chest']"/>                                                                    │
│         </object>                                                                                               │
│     </list>                                                                                                     │
│     <list name="current_meds" description="Medications the patient is currently prescribed and their            │
│ response.">                                                                                                     │
│         <object>                                                                                                │
│             <string name="medication" description="Name of the medication prescribed to the patient"/>          │
│             <string name="response" description="How is the patient responding to the prescribed medication"/>  │
│         </object>                                                                                               │
│     </list>                                                                                                     │
│ </output>                                                                                                       │
│                                                       

#### **Generating Structured data : Pydantic Model**

- Generating structured data where each column has a specific data type.
- There are 20 rows in the dataset.
- Each user has first name and last name
- The number of user orders is between 5 and 100
- Each user has a most recent order date.

- Reference : https://www.guardrailsai.com/docs/examples/generate_structured_data

In [ ]:
# Importing dependencies : guardrails, openai library , BaseModel and Field from Pydantic
import guardrails as gd
from rich import print
import openai
from pydantic import BaseModel, Field

# Importing Guardrail Validators : ValidLength, TwoWords, ValidRange
from guardrails.validators import ValidLength, TwoWords, ValidRange

# Importing List from Typing Module
from typing import List

In [ ]:
# Providing a prompt for the model and defining Pydantic Classes :
prompt = """

Generate a dataset of user orders. Each row of the dataset
should be valid

${gr.complete_json_suffix}
"""

class Order(BaseModel):
  user_id: str = Field(
      description = "The user id.",
      validators=[("1-indexed", "noop")]
  )
  user_name: str = Field(
      description = "The users first name and last name",
      validators=[TwoWords()]
  )
  num_orders: int = Field(
      description = "The number of  orders placed by the user",
      validators=[ValidRange(5, 100)]
  )
  order_date: str = Field(
      description = "The most recent order date",
  )

class Orders(BaseModel):
  user_orders: List[Order]  = Field(
      description = "Generate a list of user orders and how many orders they have placed in the past.", # with the most recent order date.
      validators=[ValidLength(20, 20, on_fail = "noop")]
  )

In [ ]:
# Defining the guard object for validating and correcting LLM response :
guard = gd.Guard.from_pydantic(output_class = Orders,
                               prompt = prompt)

In [ ]:
# Display the final prompt
print(guard.base_prompt)

Generate a dataset of user orders. Each row of the dataset
should be valid


Given below is XML that describes the information to extract from this document and the tags to extract it into.

<output>
    <list name="user_orders" description="Generate a list of user orders and how many orders they have placed in 
the past." format="length: min=20 max=20">
        <object>
            <string name="user_id" description="The user id." format="1-indexed"/>
            <string name="user_name" description="The users first name and last name" format="two-words"/>
            <integer name="num_orders" description="The number of  orders placed by the user" format="valid-range: 
min=5 max=100"/>
            <string name="order_date" description="The most recent order date"/>
        </object>
    </list>
</output>


ONLY return a valid JSON object (no other text is necessary), where the key of the field in JSON is the `name` 
attribute of the corresponding XML, and the value is of the type specified by the corresponding XML's tag. The JSON
MUST conform to the XML format, including any types and format requests e.g. requests for lists, objects and 
specific types. Be correct and concise. If you are unsure anywhere, enter `null`.

Here are examples of simple (XML, JSON) pairs that show the expected behavior:
- `<string name='foo' format='two-words lower-case' />` => `{'foo': 'example one'}`
- `<list name='bar'><string format='upper-case' /></list>` => `{"bar": ['STRING ONE', 'STRING TWO', etc.]}`
- `<object name='baz'><string name="foo" format="capitalize two-words" /><integer name="index" format="1-indexed" 
/></object>` => `{'baz': {'foo': 'Some String', 'index': 1}}`

In [ ]:
# Extracting LLM response :
raw_llm_output, validated_output, *rest = guard(
    openai.ChatCompletion.create,
    model = "gpt-4o-mini",
    max_tokens = 2048,
    temperature = 0.3
)

print(validated_output)

{
    'user_orders': [
        {'user_id': '1', 'user_name': 'John Doe', 'num_orders': 10, 'order_date': '2021-05-01'},
        {'user_id': '2', 'user_name': 'Jane Smith', 'num_orders': 5, 'order_date': '2021-05-02'},
        {'user_id': '3', 'user_name': 'Bob Johnson', 'num_orders': 20, 'order_date': '2021-05-03'},
        {'user_id': '4', 'user_name': 'Alice Williams', 'num_orders': 15, 'order_date': '2021-05-04'},
        {'user_id': '5', 'user_name': 'Mark Brown', 'num_orders': 8, 'order_date': '2021-05-05'},
        {'user_id': '6', 'user_name': 'Emily Davis', 'num_orders': 12, 'order_date': '2021-05-06'},
        {'user_id': '7', 'user_name': 'David Wilson', 'num_orders': 7, 'order_date': '2021-05-07'},
        {'user_id': '8', 'user_name': 'Karen Miller', 'num_orders': 25, 'order_date': '2021-05-08'},
        {'user_id': '9', 'user_name': 'Steven Wilson', 'num_orders': 30, 'order_date': '2021-05-09'},
        {'user_id': '10', 'user_name': 'Samantha Brown', 'num_orders': 50, 'order_date': '2021-05-10'},
        {'user_id': '11', 'user_name': 'Richard Johnson', 'num_orders': 40, 'order_date': '2021-05-11'},
        {'user_id': '12', 'user_name': 'Lauren Davis', 'num_orders': 35, 'order_date': '2021-05-12'},
        {'user_id': '13', 'user_name': 'Jacob Williams', 'num_orders': 15, 'order_date': '2021-05-13'},
        {'user_id': '14', 'user_name': 'Katie Brown', 'num_orders': 20, 'order_date': '2021-05-14'},
        {'user_id': '15', 'user_name': 'Ethan Wilson', 'num_orders': 10, 'order_date': '2021-05-15'},
        {'user_id': '16', 'user_name': 'Lily Smith', 'num_orders': 5, 'order_date': '2021-05-16'},
        {'user_id': '17', 'user_name': 'Oliver Johnson', 'num_orders': 8, 'order_date': '2021-05-17'},
        {'user_id': '18', 'user_name': 'Sophia Davis', 'num_orders': 12, 'order_date': '2021-05-18'},
        {'user_id': '19', 'user_name': 'Mia Williams', 'num_orders': 7, 'order_date': '2021-05-19'},
        {'user_id': '20', 'user_name': 'Noah Brown', 'num_orders': 15, 'order_date': '2021-05-20'}
    ]
}

In [ ]:
# Display the output using guard.history
print(guard.history.last.tree)

Logs
└── ╭────────────────────────────────────────────────── Step 0 ───────────────────────────────────────────────────╮
    │ ╭──────────────────────────────────────────────── Prompt ─────────────────────────────────────────────────╮ │
    │ │                                                                                                         │ │
    │ │                                                                                                         │ │
    │ │ Generate a dataset of user orders. Each row of the dataset                                              │ │
    │ │ should be valid                                                                                         │ │
    │ │                                                                                                         │ │
    │ │                                                                                                         │ │
    │ │ Given below is XML that describes the information to extract from this document and the tags to extract │ │
    │ │ it into.                                                                                                │ │
    │ │                                                                                                         │ │
    │ │ <output>                                                                                                │ │
    │ │     <list name="user_orders" description="Generate a list of user orders and how many orders they have  │ │
    │ │ placed in the past." format="length: min=20 max=20">                                                    │ │
    │ │         <object>                                                                                        │ │
    │ │             <string name="user_id" description="The user id." format="1-indexed"/>                      │ │
    │ │             <string name="user_name" description="The users first name and last name"                   │ │
    │ │ format="two-words"/>                                                                                    │ │
    │ │             <integer name="num_orders" description="The number of  orders placed by the user"           │ │
    │ │ format="valid-range: min=5 max=100"/>                                                                   │ │
    │ │             <string name="order_date" description="The most recent order date"/>                        │ │
    │ │         </object>                                                                                       │ │
    │ │     </list>                                                                                             │ │
    │ │ </output>                                                                                               │ │
    │ │                                                                                                         │ │
    │ │                                                                                                         │ │
    │ │ ONLY return a valid JSON object (no other text is necessary), where the key of the field in JSON is the │ │
    │ │ `name` attribute of the corresponding XML, and the value is of the type specified by the corresponding  │ │
    │ │ XML's tag. The JSON MUST conform to the XML format, including any types and format requests e.g.        │ │
    │ │ requests for lists, objects and specific types. Be correct and concise. If you are unsure anywhere,     │ │
    │ │ enter `null`.                                                                                           │ │
    │ │                                                                                                         │ │
    │ │ Here are examples of simple (XML, JSON) pairs that show the expected behavior:                          │ │
    │ │ - `<string name='foo' format='two-words lower-case' />` => `{'foo': 'example one'}`                     │ │
    │ │ - `<list name='bar'><string format='upper-c